# ST-HAE training on Kaggle GPU

Trains the **Spatial-Temporal Hierarchical Attention Ensemble** (`src/st_hae.py`) with the full
ablation on both cities (Chicago sides + NYC boroughs), on a GPU, and writes the results JSONs.

**Setup on Kaggle:** New Notebook → Settings → Accelerator = **GPU T4 x2** (or P100) → Internet = **On**.

The processed datasets are small and committed to the repo, so there is **nothing to upload** — the
clone brings the data with it. Runtime is a few minutes per city.

In [ ]:
# 1. Get the code + committed processed data
!rm -rf urban-mobility-forecasting
!git clone --depth 1 https://github.com/KC1706/urban-mobility-forecasting.git
%cd urban-mobility-forecasting

In [ ]:
# 2. Deps. torch/pandas/scikit-learn are preinstalled on Kaggle GPU images; this is a no-op safety net.
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable GPU: Settings -> Accelerator -> GPU'

In [ ]:
# 3. Chicago (9 real sides) — full model + 3-way ablation, on GPU
!python src/st_hae.py --data data/processed/chicago_taxi_sides.csv \
    --ablation --device cuda --B 2000 --out results/st_hae_chicago.json

In [ ]:
# 4. NYC (6 boroughs, 6 months) — full model + 3-way ablation, on GPU
!python src/st_hae.py --data data/processed/nyc_taxi_boroughs.csv \
    --ablation --device cuda --B 2000 --out results/st_hae_nyc.json

In [ ]:
# 5. Show the result tables (also saved as JSON under results/)
import json
for city in ['chicago', 'nyc']:
    o = json.load(open(f'results/st_hae_{city}.json'))
    rf = o['random_forest']
    print('\n' + '=' * 74)
    print(f"{city.upper()}  ({o['data']})")
    print('=' * 74)
    print(f"{'Model':20s}{'RMSE':>9s}{'R2':>9s}{'MAE':>9s}{'temporal':>11s}{'high-dmd':>10s}")
    print(f"{'RandomForest':20s}{rf['overall']['rmse']:9.2f}{rf['overall']['r2']:9.4f}"
          f"{'-':>9s}{rf['temporal_ratio_ci'][0]:10.1f}x{rf['high_demand_degradation_ci'][0]:9.0f}%")
    for v, s in o['st_hae'].items():
        ov = s['overall']
        print(f"{'ST-HAE:' + v:20s}{ov['rmse']:9.2f}{ov['r2']:9.4f}{ov['mae']:9.2f}"
              f"{s['temporal_ratio_ci'][0]:10.1f}x{s['high_demand_degradation_ci'][0]:9.0f}%")

## 6. Save results back
`results/st_hae_chicago.json` and `results/st_hae_nyc.json` are written to the notebook working
directory and appear under **Output** in the Kaggle sidebar. Download both and drop them into the
repo's `results/` folder (commit), then the paper tables in `docs/PAPER.md` §5 can be filled from
them. Each JSON contains, for RF and every ST-HAE variant: overall RMSE/R²/MAE, the temporal
worst/best-hour ratio (with 95% bootstrap CI), high-demand degradation (with CI), and per-zone R².